#  04 - Gold Layer: Sales Aggregations

In [0]:
# Databricks notebook source
# MAGIC 

from pyspark.sql.functions import *
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "delta_catalog")
dbutils.widgets.text("schema",  "delta_demo")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# COMMAND ----------
# MAGIC %md ## Read Silver tables

orders_silver = spark.table(f"{CATALOG}.{SCHEMA}.silver_orders")
customers_silver = spark.table(f"{CATALOG}.{SCHEMA}.silver_customers") \
                        .filter("is_current = true")

# COMMAND ----------
# MAGIC %md ## Broadcast join

enriched = orders_silver.join(
    broadcast(customers_silver.select("customer_id", "city", "tier")),
    "customer_id"
)

enriched.explain()
print("Enriched rows:", enriched.count())

# COMMAND ----------
# MAGIC %md ## Window aggregations

monthly_sales = enriched.groupBy("category", "year", "month") \
    .agg(
        sum("amount").alias("total_sales"),
        count("order_id").alias("order_count"),
        avg("amount").alias("avg_order_value")
    )

w_cat = Window.partitionBy("category").orderBy("year", "month")

gold_df = monthly_sales \
    .withColumn("running_total",  sum("total_sales").over(w_cat)) \
    .withColumn("prev_month_sales", lag("total_sales", 1).over(w_cat)) \
    .withColumn("mom_growth_pct",
        when(col("prev_month_sales").isNull(), None)
        .otherwise(
            round((col("total_sales") - col("prev_month_sales")) 
                  * 100 / col("prev_month_sales"), 2)
        )
    ) \
    .withColumn("category_rank",
        dense_rank().over(
            Window.partitionBy("year", "month")
                  .orderBy(col("total_sales").desc())
        )
    )

# COMMAND ----------
# MAGIC %md ## Write Gold

gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_sales_summary")

print("Gold written:", 
      spark.table(f"{CATALOG}.{SCHEMA}.gold_sales_summary").count())

gold_df.orderBy("category", "year", "month").show(20)

dbutils.notebook.exit("04_gold: SUCCESS")